# Silver Layer — Categories
## SalesFlow Data Lakehouse | Phase 4: Curated Layer

Reads `salesflow_dev.bronze.categories`, applies minimal cleaning,
and writes the curated result to `salesflow_dev.silver.categories`.

**Transformations applied:**
| Step | Transformation |
|---|---|
| 1 | Remove duplicates by `CategoryID` |
| 2 | Clean `CategoryName` and `Description` |
| 3 | Fill nulls in `Description` |
| 4 | Add `data_quality_status` flag |
| 5 | Add `processing_timestamp` |

In [0]:
%run ../04_Utils/common_functions

## 1. Read from Bronze

In [0]:
from pyspark.sql.functions import current_timestamp

df = spark.table("salesflow_dev.bronze.categories")
print(f"Records read from Bronze: {df.count()}")
display(df.limit(5))

## 2. Remove Duplicates

In [0]:
df = df.dropDuplicates(["CategoryID"])
print(f"Records after deduplication: {df.count()}")

## 3. Clean and Standardize Columns

In [0]:
# Clean text columns
df = clean_string_column(df, "CategoryName")
df = clean_string_column(df, "EnglishDescription")

# Description is informational — fill nulls with N/A
df = df.fillna({"EnglishDescription": "N/A"})

## 4. Add Quality Flag
`INVALID` if `CategoryID` or `CategoryName` is null.

In [0]:
df = add_quality_flag(df, ["CategoryID", "CategoryName"])

## 5. Add Processing Timestamp

In [0]:
df = df.withColumn("processing_timestamp", current_timestamp())

## 6. Save as Delta Table

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("salesflow_dev.silver.categories")
print("Table saved: salesflow_dev.silver.categories")

## 7. Validation

In [0]:
silver_categories = spark.table("salesflow_dev.silver.categories")
print(f"Total records: {silver_categories.count()}")
print("\nQuality flag distribution:")
display(silver_categories.groupBy("data_quality_status").count())
print("\nSchema:")
silver_categories.printSchema()
print("\nFirst 5 rows:")
display(silver_categories.limit(5))